# CellTypist Reference Annotation of Lymphoid Cells

Transfers labels from pretrained CellTypist models onto the annotated lymphoid object
(from 1.3a) as an independent check of the marker-based `cell_state` labels, for all
samples and for primary lung only (liver biopsies removed):

- **Immune_All_Low** (pan-tissue immune, high resolution; Domínguez Conde et al., Science 2022): best match
- **HLCA** (`Human_Lung_Atlas`, healthy lung; Sikkema et al., Nat Med 2023): optional, commented in config

Majority voting runs over the stored neighbourhood graph. Predictions from each model
are kept in separate, prefixed columns and saved to `tables/`.

In [ ]:
# !pip install --quiet scanpy celltypist
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/lymphoid

In [ ]:
import os

import celltypist
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from celltypist import models

## Config

In [ ]:
input_path = "sub_Lymphoid_raw_scvi_combine_annotated.h5ad"

figures_output_dir = "./figures"
tables_output_dir = "./tables"

counts_layer = "raw_counts"
reference_col = "cell_state"          # marker-based labels from 1.3a
target_sum = 1e4                      # CellTypist expects log1p of counts normalised to 10,000

# sample subsets, each annotated separately; key = filename suffix, value = obs["ID"] values to exclude
liver_biopsy_ids = ["R008", "R011"]
subsets = {"": None, "_primary_lung_only": liver_biopsy_ids}

# one entry per model; columns are written as "<prefix>predicted_labels", "<prefix>majority_voting", ...
celltypist_runs = [
    {"prefix": "immune_low_", "model": "Immune_All_Low.pkl", "mode": "best match", "p_thres": 0.5,
     "highlight": ["Regulatory T cells", "Tcm/Naive helper T cells", "Tem/Effector helper T cells",
                   "Tem/Trm cytotoxic T cells"]},
    # {"prefix": "hlca_", "model": "Human_Lung_Atlas.pkl", "mode": "best match", "p_thres": 0.5, "highlight": []},
]
majority_voting = True
umap_genes = ["CD8A", "CD8B", "CD4", "FOXP3", "IL2RA"]

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, figsize=(4, 4))

## 1. Load and normalise

In [ ]:
adata_all = sc.read(input_path)
adata_all.X = adata_all.layers[counts_layer].copy() if counts_layer in adata_all.layers else adata_all.X.copy()
sample = adata_all.X[:1000].toarray() if sp.issparse(adata_all.X) else np.asarray(adata_all.X[:1000])
assert np.allclose(sample, np.round(sample)), "expected raw counts before normalisation"

adata_all.uns.pop("log1p", None)
sc.pp.normalize_total(adata_all, target_sum=target_sum)
sc.pp.log1p(adata_all)
assert "neighbors" in adata_all.uns, "majority voting over-clusters on the stored neighbourhood graph; run sc.pp.neighbors first"
print(adata_all.n_obs, "cells;", adata_all.obs[reference_col].nunique(), f"{reference_col} labels")

## 2. Load models

In [ ]:
loaded = {}
for run in celltypist_runs:
    models.download_models(model=run["model"], force_update=False)
    loaded[run["prefix"]] = models.Model.load(model=run["model"])
    print(run["model"], "|", loaded[run["prefix"]])

## 3. Annotate each subset

Each subset is annotated independently; majority voting uses the neighbourhood graph
restricted to that subset.

In [ ]:
annotated, predictions = {}, {}
for suffix, excluded in subsets.items():
    ad = adata_all.copy() if excluded is None else adata_all[~adata_all.obs["ID"].isin(excluded)].copy()
    print(f"=== subset '{suffix or 'all samples'}': {ad.n_obs} cells")
    for run in celltypist_runs:
        pred = celltypist.annotate(ad, model=loaded[run["prefix"]], mode=run["mode"],
                                   p_thres=run["p_thres"], majority_voting=majority_voting)
        ad = pred.to_adata(insert_prob=True, prefix=run["prefix"])
        predictions[(suffix, run["prefix"])] = pred
    annotated[suffix] = ad

## 4. Predictions on the UMAP

In [ ]:
for suffix, ad in annotated.items():
    for run in celltypist_runs:
        p = run["prefix"]
        sc.pl.umap(ad, color=[reference_col, f"{p}majority_voting"] + [g for g in umap_genes if g in ad.var_names],
                   cmap="OrRd", size=30, ncols=4, wspace=0.6)
        sc.pl.umap(ad, color=f"{p}predicted_labels", title=f"{p}predicted_labels{suffix}")
        for group in run["highlight"]:
            if group in set(ad.obs[f"{p}predicted_labels"]):
                sc.pl.umap(ad, color=f"{p}predicted_labels", groups=[group], size=20, title=group)
        sc.pl.umap(ad, color=f"{p}conf_score", cmap="viridis", vmin=0, vmax=1)

## 5. Agreement with marker-based labels

Dotplots: for each `cell_state`, mean CellTypist probability (colour) and fraction of
cells (size) per predicted label. Tables: cell counts per `cell_state` × label.

In [ ]:
for (suffix, p), pred in predictions.items():
    ad = annotated[suffix]
    tag = p.rstrip("_")
    celltypist.dotplot(pred, use_as_reference=reference_col, use_as_prediction="predicted_labels",
                       title=f"{tag}{suffix}: predicted_labels", save=f"_{tag}_predicted_labels{suffix}.pdf")
    celltypist.dotplot(pred, use_as_reference=reference_col, use_as_prediction="majority_voting", cmap="Greens",
                       title=f"{tag}{suffix}: majority_voting", save=f"_{tag}{suffix}.pdf")
    for column in ("predicted_labels", "majority_voting"):
        pd.crosstab(ad.obs[reference_col], ad.obs[f"{p}{column}"]).to_csv(
            f"{tables_output_dir}/celltypist_lymphoid_{tag}{suffix}_{reference_col}_vs_{column}.csv")
    display(pd.crosstab(ad.obs[reference_col], ad.obs[f"{p}majority_voting"], normalize="index").round(2))

## 6. Save predictions

In [ ]:
for suffix, ad in annotated.items():
    label_cols = [c for run in celltypist_runs for c in
                  (f"{run['prefix']}predicted_labels", f"{run['prefix']}over_clustering",
                   f"{run['prefix']}majority_voting", f"{run['prefix']}conf_score") if c in ad.obs]
    ad.obs[[reference_col] + label_cols].to_csv(f"{tables_output_dir}/celltypist_lymphoid_predictions{suffix}.csv")
    for run in celltypist_runs:
        p = run["prefix"]
        prob_cols = [f"{p}{c}" for c in loaded[p].cell_types if f"{p}{c}" in ad.obs]
        ad.obs[prob_cols].to_csv(f"{tables_output_dir}/celltypist_lymphoid_{p.rstrip('_')}_probabilities{suffix}.csv.gz")
    print(f"saved subset '{suffix or 'all samples'}':", label_cols)